# 08 — Open Images no Colab: auditoria, download em lotes e meta de 100.000 imagens

**Use CPU.** Este notebook prepara dados; a GPU será usada no treinamento seguinte.

Mantemos o modelo antigo como referência porque o 07b ainda ficou abaixo dele na validação Home.
Aqui ampliamos os dados sem preencher planilhas ou inventar anotações.

| Decisão | Configuração inicial |
|---|---|
| Meta de imagens adicionais | 100.000 imagens válidas do treino oficial Open Images |
| Primeiro piloto | Até 5 lotes de 1.000 imagens |
| Baixar | Só IDs de imagens com caixas humanas das classes do projeto |
| Origem dos pixels | Cópia CVDF, já limitada a 1.024 pixels no lado maior |
| Armazenamento | ZIPs independentes no Drive; arquivos temporários no Colab |
| Retomada | Lotes confirmados são reutilizados |
| Anotações parciais | Cobertura preservada por imagem e classe |
| Home VAL e TEST | Permanecem no ZIP atual; não são acrescentados ao treino Open Images |

A meta depende do inventário e dos filtros. Se houver menos de 100 mil imagens elegíveis,
o relatório mostra a quantidade disponível. A presença de uma classe anotada não garante
que toda fotografia seja de um ambiente interno.

**Este notebook não treina.** Os pacotes parciais exigem um treinador que use a cobertura
dos rótulos. Usar `YOLO.train()` padrão diretamente nesses pacotes pode penalizar objetos reais
sem anotação. O piloto serve para definir a integração antes do treino e da escala completa.


## 1. Instalar as dependências

Abra `Ambiente de execução → Alterar tipo de ambiente de execução → CPU`.
Execute esta célula uma vez. Se o Colab solicitar reinício após a instalação, reinicie
e continue na célula de funções abaixo.


In [ ]:
%pip install -q "requests>=2.32,<3" "pandas>=2.2,<3" "numpy>=1.26,<3" "Pillow>=10,<13" "tqdm>=4.66,<5" "matplotlib>=3.8,<4"


## 2. Carregar as funções automáticas

Execute a célula abaixo. Ela contém as rotinas de seleção, auditoria, proteção da base,
download, compactação e retomada. As configurações que você pode mudar ficam na etapa 3.


In [ ]:
"""Preparação de Open Images no Colab; sem treino nem alteração da base Home.

Este módulo será incorporado ao notebook entregue, que é independente.
"""
from __future__ import annotations

import csv
import hashlib
import heapq
import io
import json
import math
import os
import re
import shutil
import sqlite3
import threading
import time
import zipfile
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import requests
from PIL import Image
from tqdm.auto import tqdm

VERSAO_PIPELINE = "08-openimages-1.0"
FONTES = {
    "classes": "https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions-boxable.csv",
    "hierarquia": "https://storage.googleapis.com/openimages/2018_04/bbox_labels_600_hierarchy.json",
    "caixas": "https://storage.googleapis.com/openimages/v6/oidv6-train-annotations-bbox.csv",
    "rotulos_humanos": "https://storage.googleapis.com/openimages/v5/train-annotations-human-imagelabels-boxable.csv",
    "imagens_metadata": "https://storage.googleapis.com/openimages/2018_04/train/train-images-boxable-with-rotation.csv",
}
NOMES = ["bed", "sofa", "chair", "table", "lamp", "tv", "laptop", "wardrobe",
         "window", "door", "potted plant", "photo frame"]
RAIZES_OI = ["Bed", "Couch", "Chair", "Table", "Lamp", "Television", "Laptop",
             "Wardrobe", "Window", "Door", "Houseplant", "Picture frame"]
MIDS_RAIZES = ["/m/03ssj5", "/m/02crq1", "/m/01mzpv", "/m/04bcr3", "/m/0dtln",
               "/m/07c52", "/m/01c648", "/m/02vkqh8", "/m/0d4v4", "/m/02dgv",
               "/m/03fp41", "/m/06z37_"]
# Correspondências conservadoras: toda imagem com uma destas categorias é
# excluída, em vez de omitir sua caixa ou decidir a classe pela aparência.
AMBIGUAS = {"Studio couch", "Sofa bed", "Infant bed"}
PESOS_PRIORIDADE = [2, 2, 1, 1, 1, 1, 1, 3, 1, 1, 3, 1]
TODAS_CONHECIDAS = (1 << 12) - 1
HEX_ID = re.compile(r"^[0-9a-f]{16}$")
_thread_local = threading.local()


def agora():
    return datetime.now(timezone.utc).isoformat()


def sha256_arquivo(p):
    h = hashlib.sha256()
    with Path(p).open("rb") as f:
        for b in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(b)
    return h.hexdigest()


def json_canonico(x):
    return json.dumps(x, sort_keys=True, ensure_ascii=False, separators=(",", ":"))


def salvar_json(p, x):
    p = Path(p)
    p.parent.mkdir(parents=True, exist_ok=True)
    temporario = p.with_name(p.name + ".tmp")
    temporario.write_text(json.dumps(x, ensure_ascii=False, indent=2), encoding="utf-8")
    temporario.replace(p)


def copiar_verificado(origem, destino):
    """Publica somente após comparar o hash da cópia; preserva a origem."""
    origem, destino = Path(origem), Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    digest = sha256_arquivo(origem)
    if destino.exists():
        if sha256_arquivo(destino) != digest:
            raise RuntimeError(f"Destino existente diferente: {destino}. Use outra pasta de preparação.")
        return digest
    temporario = destino.with_name(destino.name + ".uploading")
    shutil.copyfile(origem, temporario)
    if sha256_arquivo(temporario) != digest:
        raise RuntimeError(f"Cópia incompleta: {temporario}. A origem foi preservada.")
    temporario.replace(destino)
    return digest


def baixar_fonte(url, destino):
    """Download local com Range, validação de tamanho e retomada parcial."""
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    recibo = destino.with_suffix(destino.suffix + ".json")
    if destino.exists() and recibo.exists():
        r = json.loads(recibo.read_text())
        if r["url"] == url and sha256_arquivo(destino) == r["sha256"]:
            return r
        raise RuntimeError(f"Fonte local modificada: {destino}")
    parcial = destino.with_suffix(destino.suffix + ".part")
    ultimo = None
    for tentativa in range(3):
        inicio = parcial.stat().st_size if parcial.exists() else 0
        try:
            headers = {"Accept-Encoding": "identity"}
            if inicio:
                headers["Range"] = f"bytes={inicio}-"
            with requests.get(url, headers=headers,
                              stream=True, timeout=(15, 90)) as resposta:
                if resposta.status_code == 416 and inicio:
                    # Uma sessão pode terminar após gravar todos os bytes e
                    # antes de renomear o .part. Recupere esse caso completo.
                    faixa = resposta.headers.get("Content-Range", "")
                    if faixa == f"bytes */{inicio}":
                        parcial.replace(destino)
                        resultado = {"url": url, "bytes": destino.stat().st_size,
                                     "sha256": sha256_arquivo(destino), "obtido_em": agora(),
                                     "etag": resposta.headers.get("ETag")}
                        salvar_json(recibo, resultado)
                        return resultado
                resposta.raise_for_status()
                retomando = inicio > 0 and resposta.status_code == 206
                if retomando:
                    faixa = resposta.headers.get("Content-Range", "")
                    if not faixa.startswith(f"bytes {inicio}-"):
                        raise RuntimeError("Servidor devolveu intervalo inesperado.")
                    total = int(faixa.rsplit("/", 1)[1])
                else:
                    inicio = 0
                    total = int(resposta.headers.get("Content-Length", 0))
                with parcial.open("ab" if retomando else "wb") as f, tqdm(
                    total=total or None, initial=inicio, unit="B", unit_scale=True,
                    desc=destino.name, leave=False
                ) as barra:
                    for bloco in resposta.iter_content(4 * 1024 * 1024):
                        if bloco:
                            f.write(bloco)
                            barra.update(len(bloco))
                if total and parcial.stat().st_size != total:
                    raise RuntimeError("Fonte com tamanho incompleto; será retomada.")
                parcial.replace(destino)
                resultado = {"url": url, "bytes": destino.stat().st_size,
                             "sha256": sha256_arquivo(destino), "obtido_em": agora(),
                             "etag": resposta.headers.get("ETag")}
                salvar_json(recibo, resultado)
                return resultado
        except (requests.RequestException, OSError, RuntimeError) as e:
            ultimo = e
            if tentativa < 2:
                time.sleep(2 ** tentativa)
    raise RuntimeError(f"Não foi possível obter {url}: {ultimo}")


def carregar_taxonomia(classes_path, hierarquia_path):
    with Path(classes_path).open(encoding="utf-8-sig", newline="") as f:
        descricoes = {r[0]: r[1] for r in csv.reader(f) if r and r[0].startswith("/")}
    if any(descricoes.get(m) != n for m, n in zip(MIDS_RAIZES, RAIZES_OI)):
        raise RuntimeError("MIDs ou nomes oficiais incompatíveis com a especificação.")
    arvore = json.loads(Path(hierarquia_path).read_text())
    descendentes, ancestrais = defaultdict(set), defaultdict(set)

    def visitar(no, pais=()):
        mid = no["LabelName"]
        ancestrais[mid].update(pais)
        descendentes[mid].add(mid)
        for pai in pais:
            descendentes[pai].add(mid)
        for filho in no.get("Subcategory", []):
            visitar(filho, (*pais, mid))

    visitar(arvore)
    ambiguas = {m for m, n in descricoes.items() if n in AMBIGUAS}
    aliases, negativos, linhas = {}, defaultdict(int), []
    for cid, raiz in enumerate(MIDS_RAIZES):
        if raiz not in descendentes:
            raise RuntimeError(f"Classe ausente da hierarquia: {raiz}")
        # As mesas específicas e Loveseat entram na classe geral. Categorias
        # ambíguas são excluídas como imagem completa e nunca viram background.
        for mid in sorted(descendentes[raiz] - ambiguas):
            if mid in aliases and aliases[mid] != cid:
                raise RuntimeError("Subclasse pertence a dois alvos; mapping precisa ser revisto.")
            aliases[mid] = cid
            linhas.append({"id_projeto": cid, "classe_projeto": NOMES[cid],
                           "mid_openimages": mid, "classe_openimages": descricoes[mid]})
        # Uma ausência verificada da categoria ou de um ancestral implica
        # ausência do alvo; presença de ancestral não implica cobertura do alvo.
        for mid in {raiz, *ancestrais[raiz]}:
            negativos[mid] |= 1 << cid
    positivos = {mid: 1 << cid for cid, mid in enumerate(MIDS_RAIZES)}
    return {"aliases": aliases, "ambiguas": ambiguas, "negativos": dict(negativos),
            "positivos": positivos, "mapeamento": linhas, "descricoes": descricoes}


def abrir_banco(caminho):
    con = sqlite3.connect(caminho)
    con.row_factory = sqlite3.Row
    con.executescript("""
        create table if not exists images(
            id text primary key, present integer default 0,
            positive integer default 0, negative integer default 0,
            known integer default 0, metadata text);
        create table if not exists boxes(
            image_id text, cls integer, xmin real, xmax real, ymin real, ymax real,
            mid text, source text, occluded integer, truncated integer);
        create table if not exists excluded(
            image_id text, reason text, primary key(image_id,reason));
        create table if not exists schedule(
            ord integer primary key, image_id text unique,
            primary_class integer, known_count integer);
    """)
    return con


def indexar_caixas(con, caminho, taxonomia, chunk=200_000):
    aliases, ambiguas = taxonomia["aliases"], taxonomia["ambiguas"]
    colunas = ["ImageID", "LabelName", "Source", "Confidence", "XMin", "XMax", "YMin", "YMax",
               "IsGroupOf", "IsDepiction", "IsOccluded", "IsTruncated"]
    lidos = relevantes = 0
    for df in pd.read_csv(caminho, usecols=colunas, dtype={"ImageID": str, "LabelName": str}, chunksize=chunk):
        lidos += len(df)
        df = df[df.LabelName.isin(set(aliases) | ambiguas)]
        relevantes += len(df)
        presentes, caixas, excluidos = defaultdict(int), [], []
        for r in df.to_dict("records"):
            iid, mid = r["ImageID"], r["LabelName"]
            if not HEX_ID.fullmatch(iid):
                raise RuntimeError("ImageID inválido no CSV oficial.")
            if mid in ambiguas:
                excluidos.append((iid, "categoria_ambigua"))
                continue
            cid = aliases[mid]
            presentes[iid] |= 1 << cid
            xy = [float(r[k]) for k in ("XMin", "XMax", "YMin", "YMax")]
            if not (all(math.isfinite(x) for x in xy) and 0 <= xy[0] < xy[1] <= 1
                    and 0 <= xy[2] < xy[3] <= 1):
                excluidos.append((iid, "caixa_invalida"))
                continue
            if r["IsGroupOf"] != 0 or r["IsDepiction"] != 0:
                excluidos.append((iid, "grupo_depicao_ou_atributo_desconhecido"))
                continue
            if r["Confidence"] != 1:
                excluidos.append((iid, "confidence_caixa_invalida"))
                continue
            caixas.append((iid, cid, *xy, mid, r["Source"], int(r["IsOccluded"]), int(r["IsTruncated"])))
        con.executemany("insert into images(id,present) values(?,?) on conflict(id) do update set present=present|excluded.present",
                        presentes.items())
        con.executemany("insert into boxes values(?,?,?,?,?,?,?,?,?,?)", caixas)
        con.executemany("insert or ignore into excluded values(?,?)", excluidos)
        con.commit()
        print(f"Caixas lidas: {lidos:,}; registros de interesse: {relevantes:,}", end="\r", flush=True)
    con.execute("create index if not exists box_image on boxes(image_id)")
    con.execute("create index if not exists excluded_image on excluded(image_id)")
    con.commit()
    print()
    return {"linhas_caixas_lidas": lidos, "linhas_caixas_de_interesse": relevantes}


def indexar_rotulos(con, caminho, taxonomia, chunk=200_000):
    ids = {r[0] for r in con.execute("select id from images")}
    pmap, nmap = taxonomia["positivos"], taxonomia["negativos"]
    lidos = 0
    for df in pd.read_csv(caminho, usecols=["ImageID", "LabelName", "Confidence"],
                          dtype={"ImageID": str, "LabelName": str}, chunksize=chunk):
        lidos += len(df)
        df = df[df.ImageID.isin(ids) & df.LabelName.isin(set(pmap) | set(nmap))]
        pos, neg = defaultdict(int), defaultdict(int)
        for r in df.itertuples(index=False):
            iid, mid, conf = r.ImageID, r.LabelName, r.Confidence
            if conf == 1 and mid in pmap:
                pos[iid] |= pmap[mid]
            elif conf == 0 and mid in nmap:
                neg[iid] |= nmap[mid]
            elif conf not in (0, 1):
                raise RuntimeError("Rótulo humano tem Confidence diferente de 0/1.")
        con.executemany("update images set positive=positive|? where id=?", [(v, k) for k, v in pos.items()])
        con.executemany("update images set negative=negative|? where id=?", [(v, k) for k, v in neg.items()])
        con.commit()
        print(f"Rótulos humanos lidos: {lidos:,}", end="\r", flush=True)
    con.execute("update images set known=positive|negative")
    con.execute("insert or ignore into excluded select id,'negativo_contradiz_caixa_ou_positivo' from images where (negative & (present|positive)) != 0")
    con.execute("insert or ignore into excluded select id,'positivo_sem_caixa_correspondente' from images where (positive & ~present) != 0")
    con.commit()
    print()
    return {"linhas_rotulos_humanos_lidas": lidos}


def indexar_metadata(con, caminho, chunk=100_000):
    ids = {r[0] for r in con.execute("select id from images")}
    lidos = 0
    for df in pd.read_csv(caminho, dtype=str, keep_default_na=False, chunksize=chunk):
        lidos += len(df)
        df = df[df.ImageID.isin(ids)]
        atualizacoes = []
        for r in df.to_dict("records"):
            if r.get("Subset") != "train":
                raise RuntimeError("Metadata fora do treino oficial foi encontrado.")
            atualizacoes.append((json_canonico(r), r["ImageID"]))
        con.executemany("update images set metadata=? where id=?", atualizacoes)
        con.commit()
        print(f"Metadata de imagens lidos: {lidos:,}", end="\r", flush=True)
    con.execute("insert or ignore into excluded select id,'metadata_imagem_ausente' from images where metadata is null")
    # Preserve a atribuição de cada imagem; não baixar exemplos sem esse vínculo.
    con.execute("insert or ignore into excluded select id,'origem_ou_licenca_ausente' from images where json_extract(metadata,'$.License')='' or json_extract(metadata,'$.OriginalLandingURL')=''")
    for campo in ("OriginalMD5", "OriginalLandingURL"):
        grupos = defaultdict(list)
        for r in con.execute("select id,metadata from images where metadata is not null and id not in(select image_id from excluded)"):
            valor = json.loads(r["metadata"]).get(campo, "")
            if valor:
                grupos[valor].append(r["id"])
        dup = [(iid, f"duplicado_metadata_{campo}") for v in grupos.values() for iid in sorted(v)[1:]]
        con.executemany("insert or ignore into excluded values(?,?)", dup)
    con.commit()
    print()
    return {"linhas_metadata_lidas": lidos}


def criar_ordem(con, meta, seed):
    """Prioriza cobertura e distribui a seleção; uma imagem só aparece uma vez."""
    rows = list(con.execute("select id,present,positive,negative,known from images where id not in(select image_id from excluded) order by id"))
    disponibilidade = Counter(cid for r in rows for cid in range(12) if r["present"] & (1 << cid))
    pools = defaultdict(list)
    for r in rows:
        presentes = [cid for cid in range(12) if r["present"] & (1 << cid)]
        principal = min(presentes, key=lambda c: (disponibilidade[c], c))
        pools[principal].append((r["id"], int(r["known"]).bit_count()))
    for cid in pools:
        pools[cid].sort(key=lambda x: (-x[1], hashlib.sha256(f"{seed}/{x[0]}".encode()).hexdigest()))
    heap = [(1 / PESOS_PRIORIDADE[cid], cid, 0) for cid, p in pools.items() if p]
    heapq.heapify(heap)
    schedule = []
    while heap:
        _, cid, indice = heapq.heappop(heap)
        iid, cobertura = pools[cid][indice]
        schedule.append((len(schedule) + 1, iid, cid, cobertura))
        if indice + 1 < len(pools[cid]):
            heapq.heappush(heap, ((indice + 2) / PESOS_PRIORIDADE[cid], cid, indice + 1))
    con.execute("delete from schedule")
    con.executemany("insert into schedule values(?,?,?,?)", schedule)
    con.commit()
    escolhidos = {r[1] for r in schedule[:meta]}
    inventario = []
    for cid, nome in enumerate(NOMES):
        bit = 1 << cid
        inventario.append({"id": cid, "classe": nome,
            "imagens_elegiveis_com_classe": disponibilidade[cid],
            "caixas_validas_em_imagens_elegiveis": con.execute("select count(*) from boxes where cls=? and image_id not in(select image_id from excluded)", (cid,)).fetchone()[0],
            "imagens_previstas_com_classe": sum(r["id"] in escolhidos and bool(r["present"] & bit) for r in rows),
            "imagens_previstas_com_cobertura_verificada": sum(r["id"] in escolhidos and bool(r["known"] & bit) for r in rows)})
    auditoria = {"meta_imagens": meta, "imagens_candidatas_com_alvos": con.execute("select count(*) from images").fetchone()[0],
        "imagens_elegiveis_antes_download": len(rows), "meta_alcancavel_antes_download": len(rows) >= meta,
        "previstas": min(meta, len(rows)),
        "cobertura_12_classes_antes_download": sum(r["known"] == TODAS_CONHECIDAS for r in rows),
        "distribuicao_numero_classes_conhecidas": dict(sorted(Counter(int(r["known"]).bit_count() for r in rows).items())),
        "exclusoes": {r[0]: r[1] for r in con.execute("select reason,count(*) from excluded where image_id in(select id from images) group by reason")},
        "contagens_de_exclusoes_podem_se_sobrepor": True,
        "origem": "train oficial Open Images; as 12 classes permanecem nos IDs atuais",
        "semantica": "Houseplant e Picture frame precisam de conferência visual em relação ao HomeObjects",
        "relevancia": "Classe alvo anotada; filtro não comprova que toda foto seja de ambiente interno",
        "pronto_para_yolo_padrao": all(r["known"] == TODAS_CONHECIDAS for r in rows if r["id"] in escolhidos) and bool(escolhidos),
        "tratamento_rotulos_parciais": "Máscara por imagem/classe no próximo treinador; positivos de caixas sempre supervisionados. Nenhum pseudo-rótulo criado."}
    return inventario, auditoria


def fingerprint_image(raw):
    with Image.open(io.BytesIO(raw)) as im:
        im.load()
        rgb = im.convert("RGB")
        largura, altura = rgb.size
        pixel_sha = hashlib.sha256(f"{largura}x{altura}:".encode() + rgb.tobytes()).hexdigest()
        small = np.asarray(rgb.resize((32, 32), Image.Resampling.BILINEAR), dtype=np.uint8)
        gray = np.asarray(rgb.convert("L").resize((9, 8), Image.Resampling.BILINEAR))
        bits = np.packbits((gray[:, 1:] > gray[:, :-1]).reshape(-1)).tobytes().hex()
        media = small.mean(axis=(0, 1)).round(1).tolist()
        return {"width": largura, "height": altura, "pixel_sha256": pixel_sha,
                "dhash": bits, "rgb_mean": media, "sha256": hashlib.sha256(raw).hexdigest()}


def proteger_home(zip_path, sha_zip, sha_test):
    zip_path = Path(zip_path)
    if sha256_arquivo(zip_path) != sha_zip:
        raise RuntimeError("ZIP Home+SUN diferente do validado no 07b.")
    fingerprints = []
    with zipfile.ZipFile(zip_path) as z:
        nomes = z.namelist()
        if len(nomes) != len(set(nomes)):
            raise RuntimeError("O ZIP Home contém nomes duplicados.")
        for n in nomes:
            if PurePosixPath(n).is_absolute() or ".." in PurePosixPath(n).parts:
                raise RuntimeError("Caminho inseguro dentro do ZIP Home.")
        m = json.loads(z.read("verificacao_preparacao.json"))
        if {int(k): v for k, v in m["classes"].items()} != dict(enumerate(NOMES)):
            raise RuntimeError("As classes do ZIP Home+SUN são diferentes das 12 classes preservadas.")
        if m["test"]["imagens"] != 403 or m["test"]["sha256"] != sha_test:
            raise RuntimeError("Referência do TEST não corresponde às 403 imagens preservadas.")
        prefixo = m["dataset_dir"] + "/images/"
        test_names = [a["arquivo"] for a in m["arquivos"] if a["arquivo"].startswith(prefixo + "test/")]
        if len(test_names) != 403:
            raise RuntimeError("Manifesto tem contagem diferente de imagens TEST.")
        for a in tqdm(m["arquivos"], desc="Integridade Home+SUN", leave=False):
            raw = z.read(a["arquivo"])
            if hashlib.sha256(raw).hexdigest() != a["sha256"]:
                raise RuntimeError("Arquivo do Home+SUN diferente do manifesto.")
            if a["arquivo"].startswith(prefixo):
                fingerprints.append({"origem": a["arquivo"], **fingerprint_image(raw)})
    return {"zip_sha256": sha_zip, "test_sha256": sha_test, "test_imagens": 403,
            "fingerprints": fingerprints,
            "uso_test": "somente integridade e contaminação; nenhuma predição ou métrica de TEST"}


def criar_cache(local, drive_folder, config, zip_home):
    local, drive_folder = Path(local), Path(drive_folder)
    cache_drive = drive_folder / "cache_preparacao.zip"
    cfg_sha = hashlib.sha256(json_canonico(config).encode()).hexdigest()
    cache_local = local / "cache_preparacao.zip"
    diretorio = local / "indice"
    if cache_drive.exists():
        shutil.copyfile(cache_drive, cache_local)
        with zipfile.ZipFile(cache_local) as z:
            m = json.loads(z.read("cache_manifest.json"))
            if m["config_sha256"] != cfg_sha:
                raise RuntimeError("Cache tem configuração diferente. Preserve a pasta atual e use outra preparação.")
            if sha256_arquivo(zip_home) != config["home_zip_sha256"]:
                raise RuntimeError("ZIP atual do Drive difere do protegido na preparação.")
            diretorio.mkdir(parents=True, exist_ok=True)
            for item in m["arquivos"]:
                n = item["nome"]
                if PurePosixPath(n).is_absolute() or ".." in PurePosixPath(n).parts:
                    raise RuntimeError("Cache tem caminho inseguro.")
                raw = z.read(n)
                if hashlib.sha256(raw).hexdigest() != item["sha256"]:
                    raise RuntimeError("Cache incompleto ou alterado.")
                (diretorio / n).parent.mkdir(parents=True, exist_ok=True)
                (diretorio / n).write_bytes(raw)
        print("Índice e auditoria restaurados; os CSVs grandes não serão baixados outra vez.")
        return diretorio, cfg_sha
    diretorio.mkdir(parents=True, exist_ok=True)
    if sha256_arquivo(zip_home) != config["home_zip_sha256"]:
        raise RuntimeError("ZIP Home+SUN diferente do validado; os CSVs grandes não serão baixados.")
    fontes_dir = local / "fontes_originais"
    fontes_info, paths = {}, {}
    for chave, url in FONTES.items():
        destino = fontes_dir / (chave + (".json" if chave == "hierarquia" else ".csv"))
        fontes_info[chave] = baixar_fonte(url, destino)
        paths[chave] = destino
    tax = carregar_taxonomia(paths["classes"], paths["hierarquia"])
    for nome in ["banco.sqlite", "banco.sqlite-journal"]:
        p = diretorio / nome
        if p.exists():
            p.unlink()  # somente índice temporário desta preparação sem cache pronto
    con = abrir_banco(diretorio / "banco.sqlite")
    estatisticas = indexar_caixas(con, paths["caixas"], tax)
    estatisticas.update(indexar_rotulos(con, paths["rotulos_humanos"], tax))
    estatisticas.update(indexar_metadata(con, paths["imagens_metadata"]))
    inventario, auditoria = criar_ordem(con, config["meta_imagens"], config["seed"])
    con.close()
    auditoria.update(estatisticas)
    protecao = proteger_home(zip_home, config["home_zip_sha256"], config["test_sha256"])
    pd.DataFrame(inventario).to_csv(diretorio / "inventario_classes.csv", index=False, encoding="utf-8-sig")
    salvar_json(diretorio / "auditoria_anotacoes.json", auditoria)
    salvar_json(diretorio / "protecao_home.json", protecao)
    salvar_json(diretorio / "fontes.json", fontes_info)
    salvar_json(diretorio / "mapeamento.json", tax["mapeamento"])
    salvar_json(diretorio / "config.json", config)
    manifest = {"config_sha256": cfg_sha, "arquivos": []}
    for arquivo in sorted(diretorio.iterdir()):
        if arquivo.is_file():
            manifest["arquivos"].append({"nome": arquivo.name, "sha256": sha256_arquivo(arquivo)})
    with zipfile.ZipFile(cache_local, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
        for item in manifest["arquivos"]:
            z.write(diretorio / item["nome"], item["nome"])
        z.writestr("cache_manifest.json", json_canonico(manifest))
    copiar_verificado(cache_local, cache_drive)
    print("Auditoria e índice salvos. Imagens serão processadas em uma etapa separada.")
    return diretorio, cfg_sha


class Duplicatas:
    def __init__(self, home, aceitas=()):
        self.bytes = set()
        self.pixels = set()
        self.hashes_home = defaultdict(list)
        for r in home:
            self.adicionar(r)
            self.hashes_home[r["dhash"]].append(r)
        for r in aceitas:
            self.adicionar(r)

    def adicionar(self, r):
        self.bytes.add(r["sha256"])
        self.pixels.add(r["pixel_sha256"])

    def motivo(self, r):
        if r["sha256"] in self.bytes or r["pixel_sha256"] in self.pixels:
            return "duplicata_bytes_ou_pixels"
        # Coincidência perceptual forte contra os splits existentes. Essa
        # triagem é conservadora e não prova ausência de todas as fotos similares.
        for h in self.hashes_home.get(r["dhash"], []):
            if abs(r["width"] / r["height"] - h["width"] / h["height"]) <= .01:
                if max(abs(a - b) for a, b in zip(r["rgb_mean"], h["rgb_mean"])) <= 2:
                    return "semelhanca_perceptual_forte_com_home_sun"
        return None


def buscar_imagem(iid):
    if not HEX_ID.fullmatch(iid):
        raise ValueError("ImageID inválido.")
    if not hasattr(_thread_local, "session"):
        _thread_local.session = requests.Session()
    url = f"https://open-images-dataset.s3.amazonaws.com/train/{iid}.jpg"
    ultimo = None
    for tentativa in range(3):
        try:
            with _thread_local.session.get(url, stream=True, timeout=(15, 60)) as r:
                r.raise_for_status()
                buf = io.BytesIO()
                for b in r.iter_content(256 * 1024):
                    buf.write(b)
                    if buf.tell() > 20 * 1024 * 1024:
                        raise ValueError("Imagem excede o limite de download individual.")
                return {"id": iid, "raw": buf.getvalue(), "url_cvdf": url}
        except (requests.RequestException, ValueError) as e:
            ultimo = e
            if tentativa < 2:
                time.sleep(2 ** tentativa)
    return {"id": iid, "erro": str(ultimo)}


def caixas_yolo(con, iid):
    caixas = [dict(r) for r in con.execute("select * from boxes where image_id=? order by cls,xmin,ymin,xmax,ymax", (iid,))]
    unicas, linhas, vistas = [], [], set()
    for b in caixas:
        chave = (b["cls"], *(round(b[k], 8) for k in ["xmin", "xmax", "ymin", "ymax"]))
        if chave in vistas:
            continue
        vistas.add(chave)
        cx, cy = (b["xmin"] + b["xmax"]) / 2, (b["ymin"] + b["ymax"]) / 2
        w, h = b["xmax"] - b["xmin"], b["ymax"] - b["ymin"]
        linhas.append(f"{b['cls']} {cx:.8f} {cy:.8f} {w:.8f} {h:.8f}")
        unicas.append(b)
    if not linhas:
        raise RuntimeError("Imagem selecionada sem caixas válidas.")
    return unicas, ("\n".join(linhas) + "\n").encode()


def cobertura_imagem(row):
    def classes(mask):
        return [cid for cid in range(12) if int(mask) & (1 << cid)]
    return {"known_mask": row["known"], "classes_positivas_verificadas_exaustivas": classes(row["positive"]),
            "classes_negativas_verificadas": classes(row["negative"]),
            "classes_desconhecidas_para_background": classes(TODAS_CONHECIDAS ^ row["known"]),
            "classes_com_caixas_positivas": classes(row["present"]),
            "observacao": "Caixas positivas são supervisionadas mesmo quando cobertura global da classe é desconhecida."}


def validar_pacote(p, cfg_sha, completo=True):
    with zipfile.ZipFile(p) as z:
        m = json.loads(z.read("manifesto_lote.json"))
        if m["config_sha256"] != cfg_sha:
            raise RuntimeError("Lote pertence a outra configuração.")
        esperados = {"manifesto_lote.json", "README_SUPERVISAO_PARCIAL.txt"}
        for r in m["aceitas"]:
            for campo, hashcampo in [("imagem", "sha256"), ("label", "label_sha256"), ("cobertura", "cobertura_sha256")]:
                esperados.add(r[campo])
                if completo and hashlib.sha256(z.read(r[campo])).hexdigest() != r[hashcampo]:
                    raise RuntimeError("Arquivo do lote difere do manifesto.")
        nomes = z.namelist()
        if len(nomes) != len(set(nomes)) or set(nomes) != esperados:
            raise RuntimeError("Conteúdo adicional/ausente ou repetido no ZIP.")
        ids = [r["id"] for r in m["aceitas"]]
        if len(ids) != len(set(ids)):
            raise RuntimeError("Lote tem IDs repetidos.")
        return m


def publicar_lote(p_local, pasta_lotes, cfg_sha):
    p_local, pasta_lotes = Path(p_local), Path(pasta_lotes)
    m = validar_pacote(p_local, cfg_sha)
    destino = pasta_lotes / p_local.name
    digest = copiar_verificado(p_local, destino)
    commit = {**m, "arquivo": destino.name, "zip_sha256": digest,
              "zip_bytes": destino.stat().st_size, "confirmado_em": agora()}
    salvar_json(destino.with_suffix(".ok.json"), commit)
    return commit


def restaurar_lotes(pasta, cfg_sha, verificar_sha=False):
    pasta = Path(pasta)
    pasta.mkdir(parents=True, exist_ok=True)
    commits = []
    for arquivo in sorted(pasta.glob("lote_*.zip")):
        recibo = arquivo.with_suffix(".ok.json")
        if recibo.exists():
            m = json.loads(recibo.read_text())
            if m["config_sha256"] != cfg_sha or m["zip_bytes"] != arquivo.stat().st_size:
                raise RuntimeError(f"Lote alterado ou de outra configuração: {arquivo}")
            if verificar_sha and sha256_arquivo(arquivo) != m["zip_sha256"]:
                raise RuntimeError(f"Hash inválido: {arquivo}")
        else:
            # Recupera cópia completa interrompida antes de escrever o recibo.
            m = validar_pacote(arquivo, cfg_sha)
            m = {**m, "arquivo": arquivo.name, "zip_sha256": sha256_arquivo(arquivo),
                 "zip_bytes": arquivo.stat().st_size, "confirmado_em": agora()}
            salvar_json(recibo, m)
        commits.append(m)
    for recibo in pasta.glob("lote_*.ok.json"):
        if not recibo.with_name(recibo.name.replace(".ok.json", ".zip")).exists():
            raise RuntimeError(f"Recibo sem pacote: {recibo}")
    aceitas = [r for m in commits for r in m["aceitas"]]
    if len({r["id"] for r in aceitas}) != len(aceitas):
        raise RuntimeError("ID repetido entre lotes.")
    return commits


def resumo_download(commits, config, drive_folder, motivo_parada, segundos_execucao):
    aceitas = [r for m in commits for r in m["aceitas"]]
    excluidas = [r for m in commits for r in m["excluidas"]]
    falhas = [r for m in commits for r in m["falhas_download"]]
    bytes_imagens = sum(r["image_bytes"] for r in aceitas)
    bytes_zip = sum(m["zip_bytes"] for m in commits)
    n = len(aceitas)
    resumo = {"versao": VERSAO_PIPELINE, "meta_imagens": config["meta_imagens"],
        "imagens_baixadas_validas": n, "meta_concluida": n >= config["meta_imagens"],
        "lotes_confirmados": len(commits), "bytes_pacotes": bytes_zip,
        "gib_pacotes": round(bytes_zip / 1024**3, 3),
        "media_kib_por_imagem": round(bytes_imagens / n / 1024, 2) if n else None,
        "estimativa_gib_pacotes_para_meta": round(bytes_zip / n * config["meta_imagens"] / 1024**3, 2) if n else None,
        "estimativa_depende_da_distribuicao_das_proximas_imagens": True,
        "imagens_com_cobertura_das_12_classes": sum(r["known_mask"] == TODAS_CONHECIDAS for r in aceitas),
        "pronto_para_yolo_padrao": n > 0 and all(r["known_mask"] == TODAS_CONHECIDAS for r in aceitas),
        "exclusoes_download_por_motivo": dict(Counter(r["motivo"] for r in excluidas)),
        "falhas_download_acumuladas_registradas": len(falhas),
        "falhas_download_sao_retentadas_na_proxima_execucao": True,
        "parada": motivo_parada, "segundos_nesta_execucao": round(segundos_execucao, 2),
        "home_zip_sha256": config["home_zip_sha256"], "test_sha256": config["test_sha256"], "test_imagens": 403,
        "base_home_sun_modificada": False, "rotulos_gerados_por_modelo": False,
        "treinamento_executado": False,
        "deduplicacao": "MD5/origem no metadata, bytes/pixels entre aceitas e Home+SUN, triagem perceptual forte contra Home+SUN; fotos semelhantes não são garantidamente todas detectadas.",
        "proximo_treino": "Usar cobertura por classe; ignorar background desconhecido e supervisionar as caixas positivas. Comparar em Home VAL sem usar TEST para seleção."}
    distribuicao = []
    for cid, nome in enumerate(NOMES):
        bit = 1 << cid
        distribuicao.append({"id": cid, "classe": nome,
            "imagens_com_caixa": sum(cid in r["classes_com_caixa"] for r in aceitas),
            "caixas": sum(int(r["caixas_por_classe"].get(str(cid), 0)) for r in aceitas),
            "imagens_com_cobertura_verificada": sum(bool(r["known_mask"] & bit) for r in aceitas),
            "imagens_com_background_desconhecido": sum(not bool(r["known_mask"] & bit) for r in aceitas)})
    salvar_json(Path(drive_folder) / "resumo_download.json", resumo)
    pd.DataFrame(distribuicao).to_csv(Path(drive_folder) / "distribuicao_baixada.csv", index=False, encoding="utf-8-sig")
    salvar_json(Path(drive_folder) / "manifesto_pacotes.json", {
        "config_sha256": commits[0]["config_sha256"] if commits else None,
        "lotes": [{k: m[k] for k in ["arquivo", "zip_sha256", "zip_bytes"]} for m in commits],
        "imagens": n, "nomes": NOMES, "requer_supervisao_parcial": not resumo["pronto_para_yolo_padrao"]})
    return resumo, pd.DataFrame(distribuicao)


def baixar_lotes(indice, local, drive_folder, config, cfg_sha, max_lotes=5,
                 workers=8, limite_gib=30.0, verificar_sha=False, minutos_max=120,
                 buscar=buscar_imagem):
    """Um lote é confirmado no Drive antes de remover seus arquivos locais."""
    inicio = time.monotonic()
    indice, local, drive_folder = Path(indice), Path(local), Path(drive_folder)
    pasta_lotes = drive_folder / "lotes"
    commits = restaurar_lotes(pasta_lotes, cfg_sha, verificar_sha)
    aceitas = [r for m in commits for r in m["aceitas"]]
    excluidas = [r for m in commits for r in m["excluidas"]]
    feitos = {r["id"] for r in aceitas} | {r["id"] for r in excluidas}
    prot = json.loads((indice / "protecao_home.json").read_text())
    dups = Duplicatas(prot["fingerprints"], aceitas)
    con = abrir_banco(indice / "banco.sqlite")
    schedule = [dict(r) for r in con.execute("select * from schedule order by ord") if r["image_id"] not in feitos]
    total = len(aceitas)
    bytes_salvos = sum(m["zip_bytes"] for m in commits)
    meta, lote_size = config["meta_imagens"], config["imagens_por_lote"]
    cursor = 0
    parada = "limite_de_lotes_desta_execucao"
    if total >= meta:
        parada = "meta_concluida"
    for _ in range(max_lotes if total < meta else 0):
        if time.monotonic() - inicio >= minutos_max * 60:
            parada = "limite_de_tempo_desta_execucao"
            break
        if cursor >= len(schedule):
            parada = "candidatos_disponiveis_esgotados_ou_falharam_nesta_execucao"
            break
        numero = max((m["numero_lote"] for m in commits), default=0) + 1
        work = local / "lotes_temporarios" / f"lote_{numero:06d}"
        work.mkdir(parents=True, exist_ok=True)
        pacote = local / f"lote_{numero:06d}.zip"
        lote = {"versao": VERSAO_PIPELINE, "numero_lote": numero,
                "config_sha256": cfg_sha, "aceitas": [], "excluidas": [], "falhas_download": [],
                "origem_split": "train", "criado_em": agora()}
        alvo = min(lote_size, meta - total)
        falhas_seguidas = 0
        parar_por_rede = False
        with tqdm(total=alvo, desc=f"Lote {numero} — válidas") as barra, ThreadPoolExecutor(max_workers=workers) as pool:
            while len(lote["aceitas"]) < alvo and cursor < len(schedule):
                n_busca = min(workers * 2, alvo - len(lote["aceitas"]), len(schedule) - cursor)
                sub = schedule[cursor:cursor + n_busca]
                cursor += n_busca
                resultados = list(pool.map(buscar, [r["image_id"] for r in sub]))
                for info, resposta in zip(sub, resultados):
                    iid = info["image_id"]
                    if resposta.get("erro"):
                        lote["falhas_download"].append({"id": iid, "erro": resposta["erro"]})
                        falhas_seguidas += 1
                        if falhas_seguidas >= 32:
                            parar_por_rede = True
                            break
                        continue
                    raw = resposta["raw"]
                    try:
                        fp = fingerprint_image(raw)
                    except (OSError, ValueError, Image.DecompressionBombError) as e:
                        lote["falhas_download"].append({"id": iid, "erro": f"decode: {e}"})
                        falhas_seguidas += 1
                        if falhas_seguidas >= 32:
                            parar_por_rede = True
                            break
                        continue
                    falhas_seguidas = 0
                    motivo = dups.motivo(fp)
                    if min(fp["width"], fp["height"]) < config["lado_minimo"]:
                        motivo = "dimensao_muito_pequena"
                    if motivo:
                        lote["excluidas"].append({"id": iid, "motivo": motivo})
                        continue
                    row = dict(con.execute("select * from images where id=?", (iid,)).fetchone())
                    caixas, label = caixas_yolo(con, iid)
                    if any(min((b["xmax"] - b["xmin"]) * fp["width"],
                               (b["ymax"] - b["ymin"]) * fp["height"]) < 1 for b in caixas):
                        lote["excluidas"].append({"id": iid, "motivo": "caixa_inferior_a_um_pixel"})
                        continue
                    cobertura = cobertura_imagem(row)
                    cobertura_raw = json_canonico(cobertura).encode()
                    paths = {"imagem": f"images/train/oi_{iid}.jpg", "label": f"labels/train/oi_{iid}.txt",
                             "cobertura": f"coverage/train/oi_{iid}.json"}
                    for campo, conteudo in [("imagem", raw), ("label", label), ("cobertura", cobertura_raw)]:
                        p = work / paths[campo]
                        p.parent.mkdir(parents=True, exist_ok=True)
                        p.write_bytes(conteudo)
                    registro = {"id": iid, **paths, **fp,
                        "label_sha256": hashlib.sha256(label).hexdigest(),
                        "cobertura_sha256": hashlib.sha256(cobertura_raw).hexdigest(),
                        "image_bytes": len(raw), "known_mask": row["known"],
                        "classes_com_caixa": sorted({b["cls"] for b in caixas}),
                        "caixas_por_classe": dict(Counter(str(b["cls"]) for b in caixas)),
                        "caixas_originais": caixas, "metadata_origem": json.loads(row["metadata"]),
                        "url_cvdf": resposta.get("url_cvdf"),
                        "classe_primaria_selecao": info["primary_class"]}
                    lote["aceitas"].append(registro)
                    dups.adicionar(fp)
                    barra.update(1)
                if parar_por_rede:
                    break
        salvar_json(work / "manifesto_lote.json", lote)
        (work / "README_SUPERVISAO_PARCIAL.txt").write_text(
            "Este pacote contém caixas humanas em formato YOLO e coverage/train por imagem.\n"
            "Falta de anotação não indica ausência. O treinador deve mascarar perdas negativas de classes desconhecidas.\n"
            "Caixas positivas continuam supervisionadas. Não use YOLO.train padrão diretamente nestes pacotes parciais.\n"
            "Não contém Home VAL ou TEST. Fotos CVDF já têm lado maior até 1024; JPEGs preservados sem rotação adicional.\n",
            encoding="utf-8")
        # JPEG já é comprimido: armazenamento direto; textos recebem Deflate.
        with zipfile.ZipFile(pacote, "w") as z:
            for p in sorted(work.rglob("*")):
                if p.is_file():
                    z.write(p, p.relative_to(work).as_posix(),
                            compress_type=zipfile.ZIP_STORED if p.suffix == ".jpg" else zipfile.ZIP_DEFLATED,
                            compresslevel=None if p.suffix == ".jpg" else 1)
        if bytes_salvos + pacote.stat().st_size > limite_gib * 1024**3:
            parada = "limite_configurado_de_tamanho_dos_pacotes"
            print(f"Limite de {limite_gib:g} GiB atingido. Lote preservado localmente em {pacote}.")
            break
        try:
            commit = publicar_lote(pacote, pasta_lotes, cfg_sha)
        except OSError as e:
            # Os lotes anteriores continuam válidos; não apagar o lote atual.
            resumo_download(commits, config, drive_folder, "falha_ao_copiar_no_drive", time.monotonic() - inicio)
            raise RuntimeError(f"Cópia no Drive falhou. Lote local preservado: {pacote}. Verifique espaço/quota. {e}") from e
        commits.append(commit)
        total += len(commit["aceitas"])
        bytes_salvos += commit["zip_bytes"]
        resumo, _ = resumo_download(commits, config, drive_folder, "em_andamento", time.monotonic() - inicio)
        print(f"Confirmadas: {total:,}/{meta:,}; {bytes_salvos / 1024**3:.2f} GiB; estimativa para a meta: {resumo['estimativa_gib_pacotes_para_meta']} GiB.")
        # A pasta é somente deste lote temporário recém-confirmado.
        shutil.rmtree(work)
        pacote.unlink()
        if parar_por_rede:
            parada = "falhas_seguidas_de_rede_ou_imagem"
            break
        if total >= meta:
            parada = "meta_concluida"
            break
        if not commit["aceitas"] and cursor >= len(schedule):
            parada = "candidatos_esgotados_ou_falhas_de_download"
            break
    con.close()
    return resumo_download(commits, config, drive_folder, parada, time.monotonic() - inicio)


def exibir_galeria(drive_folder, max_imagens=12):
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    pasta = Path(drive_folder) / "lotes"
    arquivos = sorted(pasta.glob("lote_*.zip"))
    if not arquivos:
        print("Sem lotes confirmados para a galeria.")
        return
    with zipfile.ZipFile(arquivos[0]) as z:
        manifesto = json.loads(z.read("manifesto_lote.json"))
        amostra, classes_vistas = [], set()
        for r in manifesto["aceitas"]:
            cid = r["classe_primaria_selecao"]
            if cid not in classes_vistas:
                amostra.append(r)
                classes_vistas.add(cid)
            if len(amostra) >= max_imagens:
                break
        if not amostra:
            print("Primeiro lote registra somente falhas/exclusões.")
            return
        n = len(amostra)
        fig, axes = plt.subplots(math.ceil(n / 3), 3, figsize=(15, 4 * math.ceil(n / 3)), squeeze=False)
        for ax in axes.flat:
            ax.axis("off")
        for ax, r in zip(axes.flat, amostra):
            im = Image.open(io.BytesIO(z.read(r["imagem"]))).convert("RGB")
            ax.imshow(im)
            for b in r["caixas_originais"]:
                x, y = b["xmin"] * im.width, b["ymin"] * im.height
                w, h = (b["xmax"] - b["xmin"]) * im.width, (b["ymax"] - b["ymin"]) * im.height
                ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor="lime", linewidth=1.2))
                ax.text(x, y, NOMES[b["cls"]], fontsize=7, color="black", bbox={"facecolor": "lime", "alpha": .8, "pad": 1})
            ax.set_title(f"{NOMES[r['classe_primaria_selecao']]} | classes conhecidas: {int(r['known_mask']).bit_count()}/12", fontsize=10)
        fig.tight_layout()
        plt.show()


## 3. Montar o Drive e configurar o piloto

Reutilize o arquivo já existente:

```text
Meu Drive/TCC_Visao_Computacional/datasets/homeobjects_sunrgbd_validado.zip
```

O nome da preparação é fixo para permitir retomada na mesma pasta. **Não mude o nome
entre sessões.** Os lotes confirmados e o cache da auditoria ficam nessa pasta.

`META_IMAGENS=100_000` define o objetivo. `MAX_LOTES_POR_EXECUCAO=5` limita o primeiro
piloto a até 5.000 imagens, sem reduzir a meta. Para continuar depois do piloto, altere
somente o limite de lotes para 100; a rotina retoma as pendências e para ao atingir a meta.

`LIMITE_GIB_PACOTES` é um limite definido por você, **não uma medição do espaço livre do Drive**.
O padrão é 30 GiB. Confira sua quota e o espaço livre no Drive antes de ampliar o download.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_TCC = Path("/content/drive/MyDrive/TCC_Visao_Computacional")
ZIP_HOME = DRIVE_TCC / "datasets/homeobjects_sunrgbd_validado.zip"
NOME_PREPARACAO = "openimages_v7_tcc_100k_v1"
PASTA_DRIVE = DRIVE_TCC / "datasets" / NOME_PREPARACAO
PASTA_LOCAL = Path("/content") / NOME_PREPARACAO

META_IMAGENS = 100_000
IMAGENS_POR_LOTE = 1_000
MAX_LOTES_POR_EXECUCAO = 5  # piloto; depois, 100 para continuar até a meta
WORKERS_DOWNLOAD = 8
MINUTOS_MAX_POR_EXECUCAO = 120  # para em um limite de lote, preservando os anteriores
LIMITE_GIB_PACOTES = 30.0  # ajuste ao espaço que você reservou no Drive
VERIFICAR_SHA_PACOTES_NA_RETOMADA = False  # True confere todos os ZIPs; use antes do treino

CONFIGURACAO = {
    "versao": VERSAO_PIPELINE,
    "meta_imagens": META_IMAGENS,
    "imagens_por_lote": IMAGENS_POR_LOTE,
    "seed": 42,
    "lado_minimo": 32,
    "nomes": NOMES,
    "mids_raizes": MIDS_RAIZES,
    "ambiguas_excluidas": sorted(AMBIGUAS),
    "pesos_prioridade": PESOS_PRIORIDADE,
    "fontes": FONTES,
    "home_zip_sha256": "ada7fbee37b83f673b1b76bc57643d1fe4648941e0e8a7b895282216aed47676",
    "test_sha256": "f48be0a2b3a93e59259b1ae0d3655028951bf270ef7f4e7916a9424530951a89",
}
if not ZIP_HOME.is_file():
    raise FileNotFoundError(f"Arquivo já utilizado no 07b ausente: {ZIP_HOME}")
if not (1 <= MAX_LOTES_POR_EXECUCAO <= 200 and 1 <= WORKERS_DOWNLOAD <= 16):
    raise ValueError("Confira os limites de lotes e workers.")
if LIMITE_GIB_PACOTES <= 0 or MINUTOS_MAX_POR_EXECUCAO <= 0:
    raise ValueError("Limites de tamanho e tempo precisam ser positivos.")
PASTA_LOCAL.mkdir(parents=True, exist_ok=True)
if shutil.disk_usage(PASTA_LOCAL).free < 10 * 1024**3:
    raise RuntimeError("Reserve ao menos 10 GiB no disco temporário do Colab para fontes, cache e um lote.")
PASTA_DRIVE.mkdir(parents=True, exist_ok=True)
arquivo_config = PASTA_DRIVE / "configuracao_preparacao.json"
if arquivo_config.exists() and json.loads(arquivo_config.read_text()) != CONFIGURACAO:
    raise RuntimeError("Pasta já possui outra configuração. Preserve-a e use um novo nome de preparação.")
salvar_json(arquivo_config, CONFIGURACAO)
salvar_json(PASTA_DRIVE / "configuracao_execucao_download.json", {
    "max_lotes": MAX_LOTES_POR_EXECUCAO, "workers": WORKERS_DOWNLOAD,
    "minutos_max": MINUTOS_MAX_POR_EXECUCAO, "limite_gib": LIMITE_GIB_PACOTES,
    "verificar_sha_na_retomada": VERIFICAR_SHA_PACOTES_NA_RETOMADA,
    "python": __import__("platform").python_version(), "pandas": pd.__version__,
    "numpy": np.__version__, "requests": requests.__version__, "criado_em": agora(),
})
print("Meta:", META_IMAGENS, "| Piloto:", MAX_LOTES_POR_EXECUCAO * IMAGENS_POR_LOTE)
print("Pasta dos resultados:", PASTA_DRIVE)


## 4. Auditar as fontes e selecionar candidatos

Na primeira execução, são obtidos cerca de **3,3 GB de CSVs oficiais** no disco temporário:
caixas, rótulos humanos e metadata das imagens. As classes e a hierarquia são arquivos pequenos.
Depois da auditoria, salvamos um cache com somente o índice necessário e os relatórios.
Nas sessões seguintes, esse cache evita repetir os downloads grandes.

A página V7 oficial aponta para caixas V6, rótulos boxable V5 e metadata 2018_04; isso é
a combinação publicada para o subconjunto de detecção, não uma troca acidental de base.

A seleção considera as 12 classes e subcategorias compatíveis, dá prioridade a classes
com menos exemplos e a maior cobertura verificada. `Wardrobe` não é fundido com `Cabinetry`,
`Closet` ou `Cupboard`. Categorias ambíguas, caixas inválidas, grupos e representações de
objetos excluem a imagem inteira. Objetos reais ocluídos ou truncados permanecem.

As caixas humanas positivas são preservadas. Classes sem cobertura explícita ficam
**desconhecidas**, sem presumir ausência nem gerar caixas com o modelo antigo.

Também verificamos o ZIP Home+SUN e usamos suas imagens apenas para checar duplicidades,
incluindo o TEST, sem fazer predições ou calcular métricas de TEST.


In [ ]:
INDICE, CONFIG_SHA = criar_cache(PASTA_LOCAL, PASTA_DRIVE, CONFIGURACAO, ZIP_HOME)
AUDITORIA = json.loads((INDICE / "auditoria_anotacoes.json").read_text())
for nome in ["inventario_classes.csv", "auditoria_anotacoes.json", "mapeamento.json", "fontes.json"]:
    copiar_verificado(INDICE / nome, PASTA_DRIVE / nome)
display(pd.read_csv(INDICE / "inventario_classes.csv"))
print(json.dumps(AUDITORIA, ensure_ascii=False, indent=2))
if AUDITORIA["imagens_elegiveis_antes_download"] == 0:
    raise RuntimeError('Nenhuma imagem elegível. Consultar os relatórios para revisar o recorte antes de baixar pixels.')
if not AUDITORIA["meta_alcancavel_antes_download"]:
    print("O inventário tem menos imagens elegíveis que a meta; será usada a quantidade disponível.")
if not AUDITORIA["pronto_para_yolo_padrao"]:
    print("Rótulos parciais encontrados: o próximo treinador deverá usar a cobertura por imagem/classe.")


## 5. Baixar, conferir e empacotar os lotes

Os downloads usam somente IDs selecionados do **train oficial**. Cada lote tem até 1.000
imagens aceitas; se uma imagem falhar ou for rejeitada, a rotina busca outra candidata.
O tamanho real e uma estimativa de espaço para 100 mil aparecem depois de cada lote.

JPEGs já comprimidos são armazenados diretamente no ZIP. Caixas, cobertura e manifesto
recebem compressão. Cada pacote é conferido localmente, copiado ao Drive e conferido por
SHA-256 antes de remover os arquivos locais desse lote.

Se houver interrupção, os lotes confirmados ficam salvos. Abra o mesmo notebook,
use a mesma pasta e execute as células na ordem. O cache e os recibos recuperam o progresso.
A perda da sessão pode exigir repetir somente o lote ainda não confirmado.

Em retomadas normais, checamos configuração, recibos e tamanho dos ZIPs. O modo de hash
completo relê todos os pacotes e detecta também alterações posteriores com o mesmo tamanho.


In [ ]:
RESUMO, DISTRIBUICAO = baixar_lotes(
    INDICE, PASTA_LOCAL, PASTA_DRIVE, CONFIGURACAO, CONFIG_SHA,
    max_lotes=MAX_LOTES_POR_EXECUCAO, workers=WORKERS_DOWNLOAD,
    limite_gib=LIMITE_GIB_PACOTES,
    verificar_sha=VERIFICAR_SHA_PACOTES_NA_RETOMADA,
    minutos_max=MINUTOS_MAX_POR_EXECUCAO,
)
display(DISTRIBUICAO)
print(json.dumps(RESUMO, ensure_ascii=False, indent=2))
print("Resultados salvos em:", PASTA_DRIVE)


## 6. Conferir a conversão visualmente

A galeria escolhe automaticamente exemplos de classes diferentes do primeiro lote.
Confira se as caixas estão alinhadas e se a interpretação de planta em vaso, guarda-roupa
e porta-retrato é compatível com o projeto. Isso é uma revisão do recorte, sem preencher
planilhas e sem criar anotações de risco.

A galeria não comprova a semântica e a cobertura de todas as imagens. Fotografias parecidas
também podem escapar da triagem automática de duplicidades; o relatório descreve esse limite.


In [ ]:
exibir_galeria(PASTA_DRIVE)


## 7. Resultados para a próxima etapa

Os resultados deste notebook são documentados pelos arquivos abaixo, localizados em
`Meu Drive/TCC_Visao_Computacional/datasets/openimages_v7_tcc_100k_v1/`:

- `inventario_classes.csv` — disponibilidade e previsão por classe;
- `auditoria_anotacoes.json` — filtros, cobertura e viabilidade da meta;
- `distribuicao_baixada.csv` — contagem realmente aceita por classe;
- `resumo_download.json` — quantidade, tamanho e estimativa para a meta;
- `configuracao_preparacao.json` — seleção utilizada.

Os ZIPs grandes permanecem no Drive. `manifesto_pacotes.json` relaciona os nomes e os hashes
que o próximo notebook utilizará para montar o treinamento.

**Escala:** após analisar o piloto e conferir o espaço, mantenha a meta e a pasta;
troque `MAX_LOTES_POR_EXECUCAO` de 5 para 100 e rode as células na ordem. Ajuste
`LIMITE_GIB_PACOTES` ao espaço reservado e conserve o limite de tempo para retomada.
Antes do próximo treino, ative `VERIFICAR_SHA_PACOTES_NA_RETOMADA=True`.

**Treino seguinte:** começar com o modelo antigo, usar uma quantidade menor e auditada
de Open Images junto ao Home TRAIN, preservar a presença de exemplos Home no treinamento,
usar um treinador para rótulos parciais e escolher o checkpoint pela validação Home.
Misturas de imagens precisam preservar as máscaras ou ficar desativadas. A escala de treino
será ampliada se as métricas do detector e dos alertas melhorarem.

O download de 100 mil imagens prepara uma reserva; não define automaticamente o melhor
tamanho de treinamento nem a quantidade de épocas. O TEST de 403 imagens fica fora da seleção.

## Fontes oficiais

- [Open Images V7: downloads e formatos](https://storage.googleapis.com/openimages/web/download_v7.html)
- [Open Images: cobertura, hierarquia e avaliação](https://storage.googleapis.com/openimages/web/evaluation.html)
- [Open Images V7: descrição](https://storage.googleapis.com/openimages/web/factsfigures_v7.html)
- [CVDF: pixels e resolução das imagens](https://github.com/cvdfoundation/open-images-dataset)
- [Colab: operações do Drive e limites](https://research.google.com/colaboratory/faq.html)

As anotações originais, metadata de autoria e licença, coordenadas e hashes ficam nos
manifestos dos lotes. Para o TCC, cite Kuznetsova et al., *The Open Images Dataset V4*, IJCV, 2020,
conforme a orientação do projeto oficial.
